# 02: Agente Analyzer (notícias e setor)

O segundo agente. Ele recebe os números do YFinance no histórico e uma tarefa do Supervisor, busca notícias
e escreve 1 a 2 parágrafos ligando os números ao que está acontecendo com a empresa e ao contexto do setor.

Ele roda em uma **passagem única** em toda análise:
1. **Empresa:** notícias recentes da própria companhia combinadas aos dados quantitativos.
2. **Setor:** tendências do setor, concorrentes e fatores macroeconômicos.

A resposta vem estruturada em duas seções obrigatórias (`## Notícias da empresa` e `## Contexto do setor`).
O nó do grafo separa as duas partes e grava cada uma no seu campo (`noticias` e `contexto_setor`), eliminando
a necessidade de uma segunda chamada ao agente.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. As duas fontes de notícia

- `buscar_noticias_web`: o Gemini pesquisa no Google Search e resume, com a lista de sites consultados. É a **padrão**: mais recente e acha mais coisa.
- `buscar_noticias`: a NewsAPI, de **reserva**. Busca em português e, se vierem menos de 2 artigos, tenta em inglês.

Os dois resultados chegam dentro de `<conteudo_externo>`: é texto da internet, tratado como dado.

In [2]:
from fluxo_financeiro.tools.news import buscar_noticias, buscar_noticias_web

print(buscar_noticias_web.invoke("VALE3 Vale mineradora"))

2026-11-08 21:57:13,494 - fluxo_financeiro - INFO - [tool] buscar_noticias_web: query='VALE3 Vale mineradora'


2026-11-08 21:57:16,781 - fluxo_financeiro - INFO - [tool] buscar_noticias_web: ok (1002 chars)


<conteudo_externo origem="google_search">
A Vale (**VALE3**) reportou lucro líquido de US$ 1,255 bilhão no terceiro trimestre de 2026, uma retração de 53% na comparação anual, pressionada pela queda nos preços do minério de ferro e pela alta nos custos de frete e petróleo. Apesar do Ebitda proforma menor (US$ 3,51 bilhões), a empresa registrou fluxo de caixa livre resiliente e avanço de 2% na receita líquida. 

Em paralelo, a mineradora atualizou suas projeções institucionais na B3, mantendo a meta de produção de minério de ferro entre 335 e 345 milhões de toneladas para 2026 e estimando investimentos (Capex) na faixa de US$ 5,4 bilhões a US$ 5,7 bilhões. 

Adicionalmente, a companhia prestou esclarecimentos formais à CVM e à B3 negando a existência de deliberações ou estudos recentes sobre o cancelamento ou reestruturação do IPO da sua unidade de metais básicos (Vale Base Metals), reafirmando que permanece inalterada a sua posição anterior sobre o tema.

Fontes: bb.com.br, btgpactual.

In [3]:
print(buscar_noticias.invoke("Vale mineradora"))

2026-11-08 21:57:16,793 - fluxo_financeiro - INFO - [tool] buscar_noticias: query='Vale mineradora'


2026-11-08 21:57:17,285 - fluxo_financeiro - INFO - [tool] buscar_noticias: 5 artigos encontrados


Notícias recentes sobre 'Vale mineradora':

<conteudo_externo origem="newsapi">
1. (2026-11-06) [InfoMoney] JPMorgan vê lucro de emergentes resiliente e mantém Brasil entre mercados preferidos
   No caso do Brasil, o banco vê um cenário mais desafiador do ponto de vista macroeconômico, mas destaca que os resultados corporativos têm mostrado resistência
The post JPMorgan vê lucro de emergentes resiliente e mantém Brasil entre mercados preferidos appear…
   [Contexto adicional]: Os resultados do terceiro trimestre têm reforçado a resiliência das empresas de mercados emergentes, mesmo em meio à recente correção das ações ligadas à inteligência artificial e às incertezas macro…

2. (2026-10-30) [Globo] Ibovespa cai 18% no pós-reeleição de Lula, maior perda semanal desde março de 2020
   A liquidação de ativos engatilhada pelo resultado eleitoral manteve o Ibovespa no negativo por mais um pregão nesta sexta-feira, o quinto consecutivo. Após oscilar entre os 167.008 e os 169.185 pontos, o ín

## 2. O prompt

Repare na seção `<papel>` (pesquisa em passagem única), no `<formato_resposta>` que exige duas seções com títulos
exatos (`## Notícias da empresa` e `## Contexto do setor`) e na regra sobre conteúdo removido pela sanitização (notebook 13).
Esses títulos são o contrato entre o prompt do LLM e a função em código que separa os campos no state.

In [4]:
from fluxo_financeiro.agents.analysts import ANALYZER_PROMPT, analyzer_agent

print(ANALYZER_PROMPT)

<papel>
Você é um analista financeiro conciso. O histórico traz os dados de mercado de uma ação
(mensagem do YFinance) e a sua tarefa (a última mensagem do Supervisor).
Numa passagem só, você pesquisa as notícias recentes da empresa e o contexto do setor em que ela atua.
</papel>

<ferramentas>
- buscar_noticias_web: use primeiro. Traz o mais recente sobre a empresa e o setor, com as fontes.
- buscar_noticias: use só se buscar_noticias_web der erro ou não trouxer nada relevante.
</ferramentas>

<processo>
1. Busque as notícias recentes da empresa.
2. Busque o setor de atuação dela, usando o setor e subsetor (ou a indústria) que o YFinance trouxe.
</processo>

<formato_resposta>
Responda com duas seções, nesta ordem e com estes títulos exatos, cada um sozinho na linha:
## Notícias da empresa
1 a 2 parágrafos combinando os números do YFinance com as notícias da empresa.
## Contexto do setor
1 a 2 parágrafos sobre o setor e o posicionamento da empresa nele.
Cite fatos concretos (datas, va

## 3. A passagem única, com um histórico montado à mão

No pipeline, o histórico chega pronto. Aqui montamos o mesmo formato: o pedido do usuário, a mensagem do
YFinance (com os dados reais de agora) e a tarefa única do Supervisor, gerada por `TAREFA_ANALISTA`.
Depois da resposta, usamos `separar_secoes` para extrair cada seção para seu respectivo campo, exatamente
como o `analyzer_node` faz no grafo.

In [5]:
from langchain_core.messages import HumanMessage

from fluxo_financeiro.agents.analysts import analyzer_agent, separar_secoes
from fluxo_financeiro.graph.supervisor import TAREFA_ANALISTA
from fluxo_financeiro.tools.financial_data import buscar_dados_acao_b3

pedido = HumanMessage("Analise a ação VALE3")
dados = HumanMessage(buscar_dados_acao_b3.invoke("VALE3"), name="YFinance")
historico_inicial = [pedido, dados]

tarefa = HumanMessage(TAREFA_ANALISTA(historico_inicial), name="Supervisor")
print("Tarefa única montada em código:\n", tarefa.text, "\n")

historico = [pedido, dados, tarefa]
resp = analyzer_agent.invoke({"messages": historico})
texto_resposta = resp["messages"][-1].text
print("RESPOSTA INTEGRAL DO ANALYZER:\n", texto_resposta, "\n")

noticias, contexto_setor = separar_secoes(texto_resposta)
print("--- CAMPO NOTICIAS ---\n", noticias, "\n")
print("--- CAMPO CONTEXTO_SETOR ---\n", contexto_setor)

2026-11-08 21:57:17,296 - fluxo_financeiro - INFO - [tool] buscar_dados_acao_b3: ticker=VALE3.SA


2026-11-08 21:57:19,300 - fluxo_financeiro - INFO - [tool] buscar_dados_acao_b3: ok (2434 chars)


Tarefa única montada em código:
 Busque notícias recentes sobre a empresa no histórico. Use os dados financeiros do YFinance como contexto para identificar o ticker e o nome da empresa.
Busque informações sobre o setor de atuação da empresa (VALE3), usando o setor e subsetor (ou a indústria) informados nos dados do YFinance, e o posicionamento dela nesse setor. Foque em tendências, concorrentes principais e fatores macroeconômicos relevantes. Se os dados trazem os maiores do setor, compare a empresa com eles.
Responda com as duas seções, nesta ordem e com estes títulos: '## Notícias da empresa' e '## Contexto do setor'. 



2026-11-08 21:57:20,168 - fluxo_financeiro - INFO - [tool] buscar_noticias_web: query='Vale S.A. VALE3 noticias recentes mercado financeiro minerio de ferro'


2026-11-08 21:57:23,825 - fluxo_financeiro - INFO - [tool] buscar_noticias_web: ok (1769 chars)


RESPOSTA INTEGRAL DO ANALYZER:
 ## Notícias da empresa
A Vale S.A. (VALE3) é a maior empresa do setor de materiais básicos na B3, com um valor de mercado de R$ 248,5 bilhões. No entanto, os números recentes refletem a alta volatilidade do mercado de commodities e pressões cíclicas: a ação é negociada a R$ 64,04, figurando perto da mínima de 52 semanas (que varia entre R$ 59,42 e R$ 91,62), com um P/L elevado de 77,2x e um ROE modesto de 0,3%. O dividend yield (DY) de 6,82% continua a atrair investidores focados em proventos, mas o cenário corporativo recente tem sido marcado por desafios operacionais e financeiros.

No noticiário corporativo, a Vale reportou seu balanço trimestral registrando um lucro líquido de US$ 1,25 bilhão, queda de 53% na comparação anual, refletindo a desvalorização global do minério de ferro, a apreciação do real e o aumento nos custos logísticos e de frete. A receita líquida, por outro lado, avançou 2% para US$ 10,599 bilhões, apoiada por uma geração de fluxo 

## O que observar

- Qual tool o Analyzer usou na busca: deve ser `buscar_noticias_web`; a NewsAPI só entra se a web der erro.
- A resposta vem dividida nas duas seções obrigatórias: `## Notícias da empresa` e `## Contexto do setor`.
- A função `separar_secoes` fatia a resposta pelos títulos e remove os marcadores markdown: cada pedaço vai para o seu campo no state (`noticias` e `contexto_setor`), alimentando os guards e o Consolidador sem precisar de uma segunda passagem.
- A tarefa única do Supervisor (`TAREFA_ANALISTA`) junta as instruções da empresa e do setor numa mensagem só, orientando a pesquisa com o setor e subsetor informados pelo YFinance.
- Os parágrafos combinam fatos recentes com os números do YFinance (preço, P/L), citando fontes concretas.